# ДЗ 3. Завантаження даних та основи SQL. DQL-команди

**Датасет:** UCI Adult (Census Income), id=2 — [archive.ics.uci.edu/dataset/2/adult](https://archive.ics.uci.edu/dataset/2/adult).
Задача домену: чи перевищує річний дохід людини $50K за даними перепису США 1994 року.

Pipeline: `ucimlrepo` → CSV → `COPY FROM STDIN` → `hw3_adult_staging` (TEXT) → `INSERT INTO ... SELECT` → `hw3_adult_clean` (типи + обмеження) → аудит → EDA.

## 1. Setup і завантаження

In [1]:
import sys, subprocess, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

pkg = "pgserver" if sys.version_info < (3, 13) else "pixeltable-pgserver"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                       pkg, "jupysql", "psycopg2-binary", "sqlalchemy", "pandas", "pyarrow", "ucimlrepo"])

import pandas as pd
from sqlalchemy import create_engine, text

pgdata = Path("/tmp/hw3_pg")
if pkg == "pgserver":
    import pgserver
    pg = pgserver.get_server(pgdata, cleanup_mode="stop")
else:
    import pixeltable_pgserver
    pg = pixeltable_pgserver.get_server(pgdata, postgres_version=16)

engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 0
%config SqlMagic.feedback = 0
%sql engine

PostgreSQL 16.14 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit


### Отримання даних з офіційного джерела

`fetch_ucirepo(id=2)` повертає об'єднаний train (`adult.data`) і test (`adult.test`) — 48 842 рядки.
Pandas-обробку не робимо: зберігаємо DataFrame у CSV як є, з явним порядком колонок, щоб усі сирі значення (`?`, порожні, `>50K.`) потрапили в staging.

In [2]:
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=2)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

COLUMNS = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
           "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
           "hours-per-week", "native-country", "income"]
df = df[COLUMNS]

csv_path = Path("data/adult.csv")
csv_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(csv_path, index=False)

print(df.shape)
df.head()

(48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Staging-таблиця: усі колонки `TEXT`

In [3]:
%%sql
DROP TABLE IF EXISTS hw3_adult_clean CASCADE;
DROP TABLE IF EXISTS hw3_adult_staging CASCADE;

CREATE TABLE hw3_adult_staging (
    age_raw            TEXT,
    workclass_raw      TEXT,
    fnlwgt_raw         TEXT,
    education_raw      TEXT,
    education_num_raw  TEXT,
    marital_status_raw TEXT,
    occupation_raw     TEXT,
    relationship_raw   TEXT,
    race_raw           TEXT,
    sex_raw            TEXT,
    capital_gain_raw   TEXT,
    capital_loss_raw   TEXT,
    hours_per_week_raw TEXT,
    native_country_raw TEXT,
    income_raw         TEXT
);

SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'hw3_adult_staging'
ORDER BY ordinal_position;

column_name,data_type
age_raw,text
workclass_raw,text
fnlwgt_raw,text
education_raw,text
education_num_raw,text
marital_status_raw,text
occupation_raw,text
relationship_raw,text
race_raw,text
sex_raw,text


### `COPY FROM STDIN`

In [4]:
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur, open(csv_path, "r", encoding="utf-8", newline="") as file:
        cur.copy_expert(
            """
            COPY hw3_adult_staging (
                age_raw, workclass_raw, fnlwgt_raw, education_raw, education_num_raw,
                marital_status_raw, occupation_raw, relationship_raw, race_raw, sex_raw,
                capital_gain_raw, capital_loss_raw, hours_per_week_raw, native_country_raw, income_raw
            )
            FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ',')
            """,
            file,
        )
    raw_conn.commit()
finally:
    raw_conn.close()

print("CSV rows:", len(df))

CSV rows: 48842


In [5]:
%%sql
SELECT COUNT(*) AS loaded_rows FROM hw3_adult_staging;

loaded_rows
48842


## 2. Staging і DDL

Перед типізацією подивимось на значення цільової колонки в сирому вигляді:

In [6]:
%%sql
SELECT income_raw, COUNT(*) AS n_rows
FROM hw3_adult_staging
GROUP BY income_raw
ORDER BY income_raw;

income_raw,n_rows
<=50K,24720
<=50K.,12435
>50K,7841
>50K.,3846


Чотири варіанти замість двох: у тестовій частині UCI (`adult.test`) мітки записані з крапкою. Це одна й та сама категорія, тому крапку прибираємо, а походження рядка зберігаємо в `source_split`.

### Обґрунтування типів і правил очищення

| Колонка | Семантика | Сирий формат | Тип у clean | Очищення / обмеження |
|---|---|---|---|---|
| `person_id` | технічний surrogate key | — | `BIGINT IDENTITY PRIMARY KEY` | у джерелі немає ідентифікатора; не є бізнес-ключем |
| `age` | вік, повні роки | ціле як текст | `SMALLINT` | `TRIM → ::SMALLINT`; `NOT NULL`, `CHECK 16–100` (перепис охоплює дорослих) |
| `fnlwgt` | ваговий коефіцієнт вибірки перепису (скільки людей «представляє» рядок) | ціле до ~1.5 млн | `INTEGER` | `NOT NULL`, `CHECK > 0`; `SMALLINT` замалий |
| `education_num` | рівень освіти як порядковий код 1–16 | ціле | `SMALLINT` | `NOT NULL`, `CHECK 1–16` |
| `capital_gain`, `capital_loss` | річний прибуток/збиток від капіталу, цілі долари | ціле | `INTEGER` | `NOT NULL`, `CHECK >= 0`; копійок у джерелі немає, тому не `NUMERIC(10,2)` |
| `hours_per_week` | робочі години на тиждень | ціле | `SMALLINT` | `NOT NULL`, `CHECK 1–99` (99 — верхня межа анкети) |
| `education`, `marital_status`, `relationship`, `race` | категоріальні ознаки без пропусків | рядок | `TEXT NOT NULL` | `TRIM`; набір категорій не фіксуємо `CHECK`, щоб не ламати імпорт нових версій |
| `sex` | стать (2 значення в джерелі) | рядок | `TEXT NOT NULL` | `CHECK IN ('Male','Female')` |
| `workclass`, `occupation`, `native_country` | категоріальні ознаки з пропусками | рядок, пропуск = `?` **або** порожнє | `TEXT NULL` | `NULLIF(NULLIF(TRIM(x), ''), '?')` — обидва маркери → `NULL` |
| `income` | target: клас доходу | `<=50K`, `>50K`, `<=50K.`, `>50K.` | `TEXT NOT NULL` | `RTRIM(..., '.')`, `CHECK IN ('<=50K','>50K')` |
| `is_high_income` | target як булевий прапорець | похідне | `BOOLEAN NOT NULL` | `income = '>50K'`; табличний `CHECK` гарантує узгодженість з `income` |
| `source_split` | походження рядка в UCI | похідне з крапки в `income_raw` | `TEXT NOT NULL` | `CHECK IN ('train','test')`; потрібно для lineage і чесного train/test |

Дат, часу й грошей з центами в Adult немає, тому `TIMESTAMPTZ`/`NUMERIC` не використовуємо штучно. Рядки не фільтруємо: невдале приведення типу зупинить `INSERT`, і ми це побачимо, а не втратимо дані мовчки.

In [7]:
%%sql
CREATE TABLE hw3_adult_clean (
    person_id      BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    age            SMALLINT NOT NULL CHECK (age BETWEEN 16 AND 100),
    workclass      TEXT,
    fnlwgt         INTEGER  NOT NULL CHECK (fnlwgt > 0),
    education      TEXT     NOT NULL,
    education_num  SMALLINT NOT NULL CHECK (education_num BETWEEN 1 AND 16),
    marital_status TEXT     NOT NULL,
    occupation     TEXT,
    relationship   TEXT     NOT NULL,
    race           TEXT     NOT NULL,
    sex            TEXT     NOT NULL CHECK (sex IN ('Male', 'Female')),
    capital_gain   INTEGER  NOT NULL CHECK (capital_gain >= 0),
    capital_loss   INTEGER  NOT NULL CHECK (capital_loss >= 0),
    hours_per_week SMALLINT NOT NULL CHECK (hours_per_week BETWEEN 1 AND 99),
    native_country TEXT,
    income         TEXT     NOT NULL CHECK (income IN ('<=50K', '>50K')),
    is_high_income BOOLEAN  NOT NULL,
    source_split   TEXT     NOT NULL CHECK (source_split IN ('train', 'test')),
    CHECK (is_high_income = (income = '>50K'))
);

INSERT INTO hw3_adult_clean (
    age, workclass, fnlwgt, education, education_num, marital_status,
    occupation, relationship, race, sex, capital_gain, capital_loss,
    hours_per_week, native_country, income, is_high_income, source_split
)
WITH parsed AS (
    SELECT
        TRIM(age_raw)::SMALLINT                                   AS age,
        NULLIF(NULLIF(TRIM(workclass_raw), ''), '?')              AS workclass,
        TRIM(fnlwgt_raw)::INTEGER                                 AS fnlwgt,
        TRIM(education_raw)                                       AS education,
        TRIM(education_num_raw)::SMALLINT                         AS education_num,
        TRIM(marital_status_raw)                                  AS marital_status,
        NULLIF(NULLIF(TRIM(occupation_raw), ''), '?')             AS occupation,
        TRIM(relationship_raw)                                    AS relationship,
        TRIM(race_raw)                                            AS race,
        TRIM(sex_raw)                                             AS sex,
        TRIM(capital_gain_raw)::INTEGER                           AS capital_gain,
        TRIM(capital_loss_raw)::INTEGER                           AS capital_loss,
        TRIM(hours_per_week_raw)::SMALLINT                        AS hours_per_week,
        NULLIF(NULLIF(TRIM(native_country_raw), ''), '?')         AS native_country,
        RTRIM(TRIM(income_raw), '.')                              AS income,
        CASE WHEN TRIM(income_raw) LIKE '%.' THEN 'test' ELSE 'train' END AS source_split
    FROM hw3_adult_staging
)
SELECT
    age, workclass, fnlwgt, education, education_num, marital_status,
    occupation, relationship, race, sex, capital_gain, capital_loss,
    hours_per_week, native_country, income, income = '>50K', source_split
FROM parsed;

SELECT COUNT(*) AS inserted_rows FROM hw3_adult_clean;

inserted_rows
48842


## 3. Data-quality audit

### 3.1. Контроль кількості рядків

In [8]:
%%sql
SELECT 'staging' AS layer, COUNT(*) AS n_rows FROM hw3_adult_staging
UNION ALL
SELECT 'clean'   AS layer, COUNT(*) AS n_rows FROM hw3_adult_clean;

layer,n_rows
staging,48842
clean,48842


**Interpretation:** У staging і clean однаково по 48 842 рядки. Жоден рядок не відфільтровано: усі числові поля успішно привелися до типів і пройшли `CHECK`, а пропуски в категоріях перетворено на `NULL`, а не видалено. Це відповідає розміру UCI Adult (32 561 train + 16 281 test).

### 3.2. Перевірка дублікатів

Природного ключа в Adult немає (немає ID людини чи домогосподарства). Як business key беремо **всі 14 ознак + income** без `person_id` і `source_split`. Така перевірка виявляє лише **точні/потенційні** дублікати: дві різні людини з однаковими анкетами теоретично можливі, хоча `fnlwgt` робить збіг малоймовірним.

In [9]:
%%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (age, workclass, fnlwgt, education, education_num, marital_status, occupation,
                    relationship, race, sex, capital_gain, capital_loss, hours_per_week,
                    native_country, income)) AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT (age, workclass, fnlwgt, education, education_num, marital_status, occupation,
                    relationship, race, sex, capital_gain, capital_loss, hours_per_week,
                    native_country, income)) AS duplicate_rows
FROM hw3_adult_clean;

total_rows,distinct_business_keys,duplicate_rows
48842,48790,52


`COUNT(DISTINCT (row))` ігнорує кортежі, де є `NULL`, тому окремо рахуємо групи через `GROUP BY` (він вважає `NULL` рівними):

In [10]:
%%sql
SELECT COUNT(*) AS duplicate_groups, SUM(n - 1) AS extra_rows, MAX(n) AS max_group_size
FROM (
    SELECT COUNT(*) AS n
    FROM hw3_adult_clean
    GROUP BY age, workclass, fnlwgt, education, education_num, marital_status, occupation,
             relationship, race, sex, capital_gain, capital_loss, hours_per_week, native_country, income
    HAVING COUNT(*) > 1
) AS g;

duplicate_groups,extra_rows,max_group_size
49,52,3


**Interpretation:** Знайдено 49 груп однакових анкет (52 зайві рядки, максимум 3 копії, ≈0.1% даних). Оскільки природного ключа немає, це потенційні дублікати: з урахуванням `fnlwgt` повторення найімовірніше технічне, але не гарантовано. Рядки не видаляємо; для ML варто перевірити, чи не потрапляють копії одночасно в train і test, і за потреби дедуплікувати.

### 3.3. Null-rate у clean (7 колонок)

In [11]:
%%sql
SELECT
    ROUND(COUNT(*) FILTER (WHERE workclass      IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_workclass,
    ROUND(COUNT(*) FILTER (WHERE occupation     IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_occupation,
    ROUND(COUNT(*) FILTER (WHERE native_country IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_native_country,
    ROUND(COUNT(*) FILTER (WHERE age            IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_age,
    ROUND(COUNT(*) FILTER (WHERE education      IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_education,
    ROUND(COUNT(*) FILTER (WHERE hours_per_week IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_hours_per_week,
    ROUND(COUNT(*) FILTER (WHERE income         IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_income,
    COUNT(*) AS total_rows
FROM hw3_adult_clean;

pct_null_workclass,pct_null_occupation,pct_null_native_country,pct_null_age,pct_null_education,pct_null_hours_per_week,pct_null_income,total_rows
5.73,5.75,1.75,0.00,0.00,0.00,0.00,48842


### Спеціальні маркери у staging

In [12]:
%%sql
SELECT
    COUNT(*) FILTER (WHERE workclass_raw IS NULL)                 AS workclass_empty,
    COUNT(*) FILTER (WHERE TRIM(workclass_raw) = '?')             AS workclass_question,
    COUNT(*) FILTER (WHERE occupation_raw IS NULL)                AS occupation_empty,
    COUNT(*) FILTER (WHERE TRIM(occupation_raw) = '?')            AS occupation_question,
    COUNT(*) FILTER (WHERE native_country_raw IS NULL)            AS country_empty,
    COUNT(*) FILTER (WHERE TRIM(native_country_raw) = '?')        AS country_question,
    COUNT(*) FILTER (WHERE TRIM(income_raw) LIKE '%.')            AS income_with_dot,
    COUNT(*) FILTER (WHERE LOWER(TRIM(education_raw)) IN ('unknown', 'n/a', 'none', '')) AS education_text_markers
FROM hw3_adult_staging;

workclass_empty,workclass_question,occupation_empty,occupation_question,country_empty,country_question,income_with_dot,education_text_markers
963,1836,966,1843,274,583,16281,0


**Interpretation:** Пропуски є лише у трьох категоріальних колонках: `workclass` 5.73%, `occupation` 5.75%, `native_country` 1.75%; числові поля і target заповнені повністю. У сирих даних пропуск закодовано двома способами — `?` (1836 / 1843 / 583) і порожнім значенням (963 / 966 / 274), тому одного `NULLIF(..., '?')` недостатньо. Пропуски `occupation` не випадкові, це видно в запиті 3. Відсутність інших маркерів (`unknown`, `n/a`) не гарантує якості: помилкові, але валідні значення (наприклад, неправдиві години) так не виявити.

## 4. Розподіли та потенційні аномалії

### 4.1. Розподіл target `income`

In [13]:
%%sql
SELECT income, COUNT(*) AS n_rows,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_adult_clean
GROUP BY income
ORDER BY n_rows DESC, income
LIMIT 10;

income,n_rows,pct_rows
<=50K,37155,76.07
>50K,11687,23.93


**Interpretation:** Клас `>50K` становить лише 23.93% (11 687 рядків) — помірний дисбаланс 3:1, тому для моделі accuracy малоінформативна, краще ROC-AUC/F1 і стратифікований split.

### 4.2. Базова статистика `hours_per_week`

In [14]:
%%sql
SELECT COUNT(hours_per_week) AS n_non_null, MIN(hours_per_week) AS min_value, MAX(hours_per_week) AS max_value,
       ROUND(AVG(hours_per_week)::NUMERIC, 2) AS avg_value, ROUND(STDDEV_SAMP(hours_per_week)::NUMERIC, 2) AS sd_value
FROM hw3_adult_clean;

n_non_null,min_value,max_value,avg_value,sd_value
48842,1,99,40.42,12.39


**Interpretation:** `hours_per_week` охоплює 1–99 годин із середнім 40.42 і SD 12.39 — пік на стандартному тижні 40 годин.

### 4.3. Outlier-check

**Статистичний (3σ) для `hours_per_week`** — розподіл приблизно симетричний навколо 40, тому правило 3σ доречне. Агрегуємо за значенням, щоб бачити масштаб:

In [15]:
%%sql
WITH stats AS (
    SELECT AVG(hours_per_week) AS mean_value, STDDEV_SAMP(hours_per_week) AS sd_value
    FROM hw3_adult_clean
    WHERE hours_per_week IS NOT NULL
)
SELECT t.hours_per_week,
       COUNT(*) AS n_rows,
       ROUND(((t.hours_per_week - s.mean_value) / NULLIF(s.sd_value, 0))::NUMERIC, 2) AS z_score
FROM hw3_adult_clean AS t
CROSS JOIN stats AS s
WHERE t.hours_per_week IS NOT NULL
  AND s.sd_value > 0
  AND ABS(t.hours_per_week - s.mean_value) > 3 * s.sd_value
GROUP BY t.hours_per_week, s.mean_value, s.sd_value
ORDER BY t.hours_per_week;

hours_per_week,n_rows,z_score
1,27,-3.18
2,53,-3.10
3,59,-3.02
78,13,3.03
79,1,3.11
80,210,3.19
81,3,3.27
82,1,3.36
84,72,3.52
85,17,3.60


**Доменний: `capital_gain = 99999`** — підозра на технічну стелю (top-coding) у джерелі:

In [16]:
%%sql
SELECT capital_gain, COUNT(*) AS n_rows,
       ROUND(AVG(is_high_income::INT) * 100, 2) AS pct_high_income
FROM hw3_adult_clean
WHERE capital_gain >= 25000
GROUP BY capital_gain
ORDER BY capital_gain DESC
LIMIT 10;

capital_gain,n_rows,pct_high_income
99999,244,100.00
41310,3,0.00
34095,6,0.00
27828,58,100.00
25236,14,100.00
25124,6,100.00


**Interpretation:** Межі 3σ: 3.25 і 77.6 години, за ними 23 значення, разом 681 рядок (≈1.4%): 1–3 години (139 рядків) і 78–99 годин (542). Найбільші групи — 80 год (210 рядків) і 99 год (137). Це правдоподібні рідкісні випадки (фермери, самозайняті, кілька робіт), тому їх **залишаємо** і за потреби додаємо прапорець; 99 може бути верхньою межею анкети, що варто **передати на перевірку джерела**. Доменна перевірка показала 244 рядки з `capital_gain = 99999` при наступному значенні 41 310: це top-coding, а не реальна сума. Рекомендація — **додати прапорець** `capital_gain_capped` і не трактувати 99 999 як точне число; усі ці рядки мають `>50K`.

## 5. Десять DQL/EDA-запитів

### Запит 1

**Business-question:** Яка частка доходу >50K у різних рівнях `education`?

In [17]:
%%sql
SELECT education, education_num,
       COUNT(*) AS n_rows,
       ROUND(AVG(is_high_income::INT) * 100, 2) AS pct_high_income
FROM hw3_adult_clean
GROUP BY education, education_num
ORDER BY education_num DESC;

education,education_num,n_rows,pct_high_income
Doctorate,16,594,72.56
Prof-school,15,834,73.98
Masters,14,2657,54.91
Bachelors,13,8025,41.28
Assoc-acdm,12,1601,25.80
Assoc-voc,11,2061,25.33
Some-college,10,10878,18.96
HS-grad,9,15784,15.86
12th,8,657,7.31
11th,7,1812,5.08


**Interpretation:** Частка високого доходу майже монотонно зростає з `education_num`: від 1.20% у Preschool і 15.86% у HS-grad до 41.28% у Bachelors та 72–74% у Doctorate/Prof-school. Різкий стрибок відбувається між Some-college (18.96%) і Bachelors, тож освіта — один із найсильніших предикторів. Найчисленніші групи — HS-grad і Some-college, де високий дохід мають менше 20%.

### Запит 2

**Business-question:** Чи відповідає кожен текстовий рівень `education` рівно одному коду `education_num`?

In [18]:
%%sql
SELECT DISTINCT education, education_num
FROM hw3_adult_clean
ORDER BY education_num;

education,education_num
Preschool,1
1st-4th,2
5th-6th,3
7th-8th,4
9th,5
10th,6
11th,7
12th,8
HS-grad,9
Some-college,10


**Interpretation:** Кожен із 16 текстових рівнів має рівно один код `education_num` від 1 до 16 без конфліктів. Отже `education` і `education_num` — дві форми однієї інформації. У моделі варто залишити одну з них (зручніше порядковий `education_num`), щоб не дублювати ознаку.

### Запит 3

**Business-question:** Які `workclass` мають найбільше пропусків `occupation` (пропущений workclass показуємо окремою групою)?

In [19]:
%%sql
SELECT COALESCE(workclass, '(missing)') AS workclass_group,
       COUNT(*) AS n_rows,
       COUNT(*) FILTER (WHERE occupation IS NULL) AS n_occupation_null,
       ROUND(COUNT(*) FILTER (WHERE occupation IS NULL) * 100.0 / COUNT(*), 2) AS pct_occupation_null
FROM hw3_adult_clean
GROUP BY COALESCE(workclass, '(missing)')
ORDER BY n_occupation_null DESC, n_rows DESC;

workclass_group,n_rows,n_occupation_null,pct_occupation_null
(missing),2799,2799,100.00
Never-worked,10,10,100.00
Private,33906,0,0.00
Self-emp-not-inc,3862,0,0.00
Local-gov,3136,0,0.00
State-gov,1981,0,0.00
Self-emp-inc,1695,0,0.00
Federal-gov,1432,0,0.00
Without-pay,21,0,0.00


**Interpretation:** Пропуски `occupation` трапляються лише у двох групах: у всіх 2 799 рядках з невідомим `workclass` (100%) і в 10 рядках `Never-worked` (100%). У решті категорій пропусків 0. Отже це структурний пропуск: для тих, хто ніколи не працював, — логічно очікуваний, а для `(missing)` — це одночасна відмова від відповіді на два питання, яку краще кодувати окремою категорією, а не імпутувати.

### Запит 4

**Business-question:** Як змінюється частка доходу >50K за діапазонами `hours_per_week`?

In [20]:
%%sql
SELECT CASE
           WHEN hours_per_week BETWEEN 1  AND 19 THEN '01-19'
           WHEN hours_per_week BETWEEN 20 AND 34 THEN '20-34'
           WHEN hours_per_week BETWEEN 35 AND 40 THEN '35-40'
           WHEN hours_per_week BETWEEN 41 AND 50 THEN '41-50'
           ELSE '51-99'
       END AS hours_band,
       COUNT(*) AS n_rows,
       ROUND(AVG(is_high_income::INT) * 100, 2) AS pct_high_income
FROM hw3_adult_clean
GROUP BY hours_band
ORDER BY hours_band;

hours_band,n_rows,pct_high_income
01-19,2591,7.29
20-34,5804,6.74
35-40,26095,20.56
41-50,8917,39.46
51-99,5435,40.90


**Interpretation:** Частка високого доходу зростає з тривалістю робочого тижня: 6–7% при менше ніж 35 годинах, 20.56% у стандартному діапазоні 35–40 і близько 40% при понад 40 годинах. Після 41–50 год приріст майже зникає (39.46% → 40.90%), тож залежність нелінійна з насиченням. Діапазон 35–40 містить більше половини всіх рядків.

### Запит 5

**Business-question:** Які `native_country` трапляються найрідше?

In [21]:
%%sql
SELECT native_country, COUNT(*) AS n_rows
FROM hw3_adult_clean
WHERE native_country IS NOT NULL
GROUP BY native_country
ORDER BY n_rows ASC, native_country
LIMIT 10;

native_country,n_rows
Holand-Netherlands,1
Hungary,19
Honduras,20
Scotland,21
Laos,23
Outlying-US(Guam-USVI-etc),23
Yugoslavia,23
Trinadad&Tobago,27
Cambodia,28
Hong,30


**Interpretation:** Найрідша країна — Holand-Netherlands (1 рядок); ще дев'ять країн мають лише 19–30 записів (Hungary, Honduras, Scotland, Laos та ін.). Такі категорії майже не несуть статистичного сигналу і можуть створити нестабільні one-hot ознаки, тому їх доцільно об'єднати в `Other`. Також помітне дивне кодування (`Hong`, `Trinadad&Tobago`, `Outlying-US(Guam-USVI-etc)`).

### Запит 6

**Business-question:** Чи відрізняється частка високого доходу у людей, народжених поза США (включно з невідомою країною), від уродженців США?

In [22]:
%%sql
SELECT CASE
           WHEN native_country IS DISTINCT FROM 'United-States' THEN 'not US / unknown'
           ELSE 'United-States'
       END AS origin,
       COUNT(*) AS n_rows,
       COUNT(*) FILTER (WHERE native_country IS NULL) AS n_unknown,
       ROUND(AVG(is_high_income::INT) * 100, 2) AS pct_high_income
FROM hw3_adult_clean
GROUP BY origin
ORDER BY origin;

origin,n_rows,n_unknown,pct_high_income
United-States,43832,0,24.40
not US / unknown,5010,857,19.82


**Interpretation:** Уродженці США мають частку високого доходу 24.40%, а група «не США або невідомо» — 19.82% (5 010 рядків, з них 857 з невідомою країною). `IS DISTINCT FROM` гарантує, що рядки з `NULL` потрапили в другу групу, а не зникли, як було б із `<>`. Різниця помітна, але менша, ніж за освітою чи годинами.

### Запит 7

**Business-question:** Скільки людей мають «нетиповий» профіль високого доходу: >50K, без диплома бакалавра чи вищого, і при цьому молодші 25 років або працюють менше 30 годин?

In [23]:
%%sql
SELECT sex, COUNT(*) AS n_rows, ROUND(AVG(age), 1) AS avg_age, ROUND(AVG(hours_per_week), 1) AS avg_hours
FROM hw3_adult_clean
WHERE is_high_income
  AND NOT (education IN ('Bachelors', 'Masters', 'Doctorate', 'Prof-school'))
  AND (age < 25 OR hours_per_week < 30)
GROUP BY sex
ORDER BY n_rows DESC;

sex,n_rows,avg_age,avg_hours
Male,141,45.5,27.9
Female,116,42.4,20.2


**Interpretation:** Таких людей лише 257 (141 чоловік і 116 жінок) із 11 687 з високим доходом, тобто близько 2%. Середній вік 42–46 років і короткий тиждень 20–28 годин вказують, що це переважно не молодь, а досвідчені працівники з частковою зайнятістю, імовірно з капітальним доходом. Це рідкісний сегмент, який модель може класифікувати гірше.

### Запит 8

**Business-question:** Як відрізняється частка високого доходу між державними (`*gov*`) і самозайнятими (`Self-emp*`) категоріями?

In [24]:
%%sql
SELECT workclass,
       COUNT(*) AS n_rows,
       ROUND(AVG(is_high_income::INT) * 100, 2) AS pct_high_income
FROM hw3_adult_clean
WHERE workclass ILIKE '%gov%' OR workclass ILIKE 'self-emp%'
GROUP BY workclass
ORDER BY pct_high_income DESC;

workclass,n_rows,pct_high_income
Self-emp-inc,1695,55.34
Federal-gov,1432,39.18
Local-gov,3136,29.56
Self-emp-not-inc,3862,27.89
State-gov,1981,26.75


**Interpretation:** Серед державних і самозайнятих категорій найвища частка високого доходу у `Self-emp-inc` — 55.34%, далі `Federal-gov` 39.18%. `Self-emp-not-inc` (27.89%) і `State-gov` (26.75%) близькі до середнього рівня. Тобто важливий не лише сектор, а й форма самозайнятості: інкорпоровані підприємці заробляють значно більше.

### Запит 9

**Business-question:** Хто ці 10 людей з найбільшим `capital_gain` та найдовшим тижнем — чи виглядають записи правдоподібно?

In [25]:
%%sql
SELECT person_id, age, workclass, occupation, education, hours_per_week, capital_gain, income
FROM hw3_adult_clean
ORDER BY capital_gain DESC, hours_per_week DESC, person_id
LIMIT 10;

person_id,age,workclass,occupation,education,hours_per_week,capital_gain,income
45930,42,Self-emp-inc,Prof-specialty,Prof-school,99,99999,>50K
48630,59,Self-emp-inc,Exec-managerial,Prof-school,84,99999,>50K
6528,49,Self-emp-inc,Prof-specialty,Masters,80,99999,>50K
16424,50,Self-emp-inc,Prof-specialty,Prof-school,80,99999,>50K
20284,49,State-gov,Sales,Masters,80,99999,>50K
23468,42,Self-emp-not-inc,Sales,Masters,80,99999,>50K
32919,50,Self-emp-not-inc,Prof-specialty,Prof-school,80,99999,>50K
38335,42,Private,Prof-specialty,Prof-school,80,99999,>50K
38857,62,Self-emp-inc,Prof-specialty,Doctorate,80,99999,>50K
39085,48,Private,Sales,Bachelors,80,99999,>50K


**Interpretation:** Усі 10 записів мають `capital_gain = 99999` і дохід `>50K`; це люди 42–62 років із Prof-school/Masters/Doctorate, найчастіше `Self-emp-inc` (4 з 10) і професія `Prof-specialty` (6 з 10), які працюють 80–99 годин. Профіль правдоподібний, але однакове значення 99 999 підтверджує top-coding, тому реальну суму прибутку ми не знаємо.

### Запит 10

**Business-question:** Яка середня сума `capital_gain` серед тих, хто його мав, і як часто він ненульовий у кожному класі доходу?

In [26]:
%%sql
SELECT income,
       COUNT(*) AS n_rows,
       COUNT(NULLIF(capital_gain, 0)) AS n_with_gain,
       ROUND(COUNT(NULLIF(capital_gain, 0)) * 100.0 / COUNT(*), 2) AS pct_with_gain,
       ROUND(AVG(capital_gain), 0) AS avg_gain_all,
       ROUND(AVG(NULLIF(capital_gain, 0)), 0) AS avg_gain_nonzero
FROM hw3_adult_clean
GROUP BY income
ORDER BY income;

income,n_rows,n_with_gain,pct_with_gain,avg_gain_all,avg_gain_nonzero
<=50K,37155,1544,4.16,147,3538
>50K,11687,2491,21.31,4042,18965


**Interpretation:** Ненульовий `capital_gain` мають 21.31% людей з `>50K` і лише 4.16% з `<=50K`; середня сума серед тих, хто його має, — 18 965 vs 3 538. `NULLIF(capital_gain, 0)` відкидає нулі з середнього, і видно, що звичайне середнє (147 і 4 042) приховує реальний масштаб. Для ML краще дві ознаки: прапорець наявності прибутку та логарифм суми.

## 6. Reflection

**Якість даних.** Пропуски є лише в `workclass`, `occupation`, `native_country` (1.7–5.8%), і закодовані двома маркерами — `?` та порожнім значенням. Пропуски `occupation` структурні: вони збігаються з невідомим `workclass` і `Never-worked`. Target у test-частині мав крапку (`>50K.`) — фактично чотири класи замість двох. Знайдено 52 потенційні дублікати. `capital_gain = 99999` — технічна стеля, а `hours_per_week = 99`, ймовірно, верхня межа анкети. Є дивні назви країн (`Hong`, `Holand-Netherlands`).

**Features.** Використав би `age`, `education_num`, `workclass`, `occupation`, `marital_status`, `relationship`, `hours_per_week`, `capital_gain`/`capital_loss` (прапорець + log), `native_country` з об'єднанням рідкісних категорій. Не використав би `person_id` (технічний ключ), `education` (дублює `education_num`), `source_split` (метадані) і `fnlwgt` як ознаку — це ваги вибірки, які краще застосувати як `sample_weight`. `race` і `sex` потребують обережності через ризик дискримінаційної моделі.

**Leakage.** Явних «майбутніх» полів немає, але `capital_gain` і `capital_loss` — частина того ж річного доходу, який ми прогнозуємо; якщо прогноз робиться до кінця року, вони недоступні. Також `relationship` (`Husband`/`Wife`) майже дублює `marital_status` і `sex`.

**Pipeline.** Попросив би data engineer уніфікувати маркер пропуску в `NULL`, прибрати крапку з target, додати ID респондента і колонку `split`, документувати top-coding і словник категорій, а також перевіряти схему, типи й діапазони ще під час експорту, щоб некоректні значення не доходили до аналітичного шару.

**ML-задача** — бінарна classification (`income > 50K`) з дисбалансом 1:3.